<a href="https://colab.research.google.com/github/yoshevijay/Product-review-acknowledgement-/blob/main/Customer_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [92]:
import pandas as pd
import re
import random
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import spacy
from spacy.lang.en.stop_words import STOP_WORDS as stopwords

In [93]:
data = pd.read_csv('archive.csv')

In [94]:
import re
stopwords = set(stopwords)
if 'Not' in stopwords:
    stopwords.remove('Not')
data['ReviewText'] = data['ReviewText'].apply(lambda x: ' '.join([t for t in x.split() if t not in stopwords]))
data['ReviewText'] = data['ReviewText'].apply(lambda x: re.sub(r'[^\w]', " ", x))

In [95]:
X = data['ReviewText']
Y = data['Rating']

In [96]:
pipelines = {
    'SVM': Pipeline([('tfidf', TfidfVectorizer()), ('clf', SVC(C=1000, gamma='auto'))]),
    'Decision Tree': Pipeline([('tfidf', TfidfVectorizer()), ('clf', DecisionTreeClassifier())]),
    'Random Forest': Pipeline([('tfidf', TfidfVectorizer()), ('clf', RandomForestClassifier())]),
    'Logistic Regression': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())])
}

In [97]:

# Compare classifiers using 5-fold cross-validation
cv_scores = {}

for name, clf in pipelines.items():
    scores = cross_val_score(clf, X, Y, cv=5)
    cv_scores[name] = scores.mean()
    print(f"{name} Cross-validated Accuracy: {scores.mean():.4f}")

# Select the best classifier
best_clf_name = max(cv_scores, key=cv_scores.get)
best_clf = pipelines[best_clf_name]

print(f"\nBest Classifier: {best_clf_name}")

# Split data into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Train the best classifier
best_clf.fit(X_train, Y_train)

# Make predictions on validation data
Y_predict = best_clf.predict(X_val)

# Evaluate model performance
print("\nConfusion Matrix:")
print(confusion_matrix(Y_val, Y_predict))

print("\nClassification Report:")
print(classification_report(Y_val, Y_predict))

print("\nValidation Accuracy:")
print(accuracy_score(Y_val, Y_predict))


SVM Cross-validated Accuracy: 0.5860
Decision Tree Cross-validated Accuracy: 0.5230
Random Forest Cross-validated Accuracy: 0.5500
Logistic Regression Cross-validated Accuracy: 0.5870

Best Classifier: Logistic Regression

Confusion Matrix:
[[18 19  2  0  0]
 [15 21  5  0  0]
 [ 0  0 40  0  0]
 [ 0  0  3 22 16]
 [ 0  0  1 12 26]]

Classification Report:
              precision    recall  f1-score   support

           1       0.55      0.46      0.50        39
           2       0.53      0.51      0.52        41
           3       0.78      1.00      0.88        40
           4       0.65      0.54      0.59        41
           5       0.62      0.67      0.64        39

    accuracy                           0.64       200
   macro avg       0.62      0.64      0.63       200
weighted avg       0.62      0.64      0.63       200


Validation Accuracy:
0.635


In [98]:
clf.predict(['I did not like the session conducted'])

array([3])

**ASPECT BASED SENTIMENT ANALYSIS**

In [99]:
def split_by_bar(string_text):
    return string_text.split(' ') #split data (=tokenize)

data['tokens'] = data['ReviewText'].apply(split_by_bar)

In [100]:
data['tokens']
data['tokens'][10]

['awful', '', 'customer', 'service', 'bad', '', 'pros', 'cons', '']

In [101]:
data['tokens']

,tokens
0,"[fantastic, , wonderful, experience, ]"
1,"[broke, easily, , awful, ]"
2,"[met, expectations, , stars, ]"
3,"[satisfied, , wonderful, experience, ]"
4,"[worst, purchase, , star, ]"
...,...
995,"[average, product, , worst, purchase, ]"
996,"[met, expectations, , poor, quality, ]"
997,"[works, expected, , stars, ]"
998,"[fantastic, , excellent, quality, , good, bad, ]"


In [102]:
data.head()

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens
0,REV2000,Product_E,User_114,4,fantastic wonderful experience,2023-04-17,"[fantastic, , wonderful, experience, ]"
1,REV2001,Product_C,User_186,2,broke easily awful,2023-11-27,"[broke, easily, , awful, ]"
2,REV2002,Product_E,User_101,3,met expectations stars,2023-12-10,"[met, expectations, , stars, ]"
3,REV2003,Product_A,User_175,5,satisfied wonderful experience,2023-11-10,"[satisfied, , wonderful, experience, ]"
4,REV2004,Product_C,User_158,1,worst purchase star,2024-05-25,"[worst, purchase, , star, ]"


In [103]:
train, dev = np.split(data, [int(len(data)*0.8)])

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [104]:
dev.shape

(200, 7)

In [105]:
train.shape

(800, 7)

In [106]:
train

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens
0,REV2000,Product_E,User_114,4,fantastic wonderful experience,2023-04-17,"[fantastic, , wonderful, experience, ]"
1,REV2001,Product_C,User_186,2,broke easily awful,2023-11-27,"[broke, easily, , awful, ]"
2,REV2002,Product_E,User_101,3,met expectations stars,2023-12-10,"[met, expectations, , stars, ]"
3,REV2003,Product_A,User_175,5,satisfied wonderful experience,2023-11-10,"[satisfied, , wonderful, experience, ]"
4,REV2004,Product_C,User_158,1,worst purchase star,2024-05-25,"[worst, purchase, , star, ]"
...,...,...,...,...,...,...,...
795,REV2795,Product_E,User_157,4,exceeded expectations best purchase ever,2023-12-30,"[exceeded, expectations, , best, purchase, eve..."
796,REV2796,Product_B,User_167,3,better star,2023-04-08,"[better, , star, ]"
797,REV2797,Product_A,User_156,2,difficult use broke easily,2023-03-26,"[difficult, use, , broke, easily, ]"
798,REV2798,Product_D,User_150,4,love product amazing features met expectations,2024-09-22,"[love, product, , amazing, features, , met, ex..."


In [107]:
dev

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens
800,REV2800,Product_E,User_102,2,worst purchase star works expected,2023-12-05,"[worst, purchase, , star, , works, expected, ]"
801,REV2801,Product_E,User_192,4,works perfectly fantastic special,2023-05-27,"[works, perfectly, , fantastic, , special, ]"
802,REV2802,Product_C,User_184,4,wonderful experience fantastic,2024-03-26,"[wonderful, experience, , fantastic, ]"
803,REV2803,Product_C,User_108,5,wonderful experience great value good bad,2023-11-30,"[wonderful, experience, , great, value, , good..."
804,REV2804,Product_A,User_121,2,terrible product waste money,2024-02-12,"[terrible, product, , waste, money, ]"
...,...,...,...,...,...,...,...
995,REV2995,Product_D,User_133,3,average product worst purchase,2024-12-10,"[average, product, , worst, purchase, ]"
996,REV2996,Product_B,User_143,3,met expectations poor quality,2023-03-19,"[met, expectations, , poor, quality, ]"
997,REV2997,Product_C,User_185,3,works expected stars,2023-03-15,"[works, expected, , stars, ]"
998,REV2998,Product_A,User_106,5,fantastic excellent quality good bad,2023-09-26,"[fantastic, , excellent, quality, , good, bad, ]"


In [108]:
from collections import Counter
from nltk.util import ngrams
n_gram = 1

In [109]:
def unigram(lst):
  unigram = Counter(ngrams(lst, n_gram))
  return unigram

In [110]:
train['ug'] = (train['tokens']).apply(unigram)

In [111]:
train['ug']

,ug
0,"{('fantastic',): 1, ('',): 2, ('wonderful',): ..."
1,"{('broke',): 1, ('easily',): 1, ('',): 2, ('aw..."
2,"{('met',): 1, ('expectations',): 1, ('',): 2, ..."
3,"{('satisfied',): 1, ('',): 2, ('wonderful',): ..."
4,"{('worst',): 1, ('purchase',): 1, ('',): 2, ('..."
...,...
795,"{('exceeded',): 1, ('expectations',): 1, ('',)..."
796,"{('better',): 1, ('',): 2, ('star',): 1}"
797,"{('difficult',): 1, ('use',): 1, ('',): 2, ('b..."
798,"{('love',): 1, ('product',): 1, ('',): 3, ('am..."


In [112]:
def featurize(token_list):
  features = {} # add output into dict = dict of features = list of feature dict
  for i in token_list:
    features[i] = 1
  return features

In [113]:
feature_dict_list = [featurize(x) for x in train['ug']]

In [114]:
feature_dict_list

[{('fantastic',): 1, ('',): 1, ('wonderful',): 1, ('experience',): 1},
 {('broke',): 1, ('easily',): 1, ('',): 1, ('awful',): 1},
 {('met',): 1, ('expectations',): 1, ('',): 1, ('stars',): 1},
 {('satisfied',): 1, ('',): 1, ('wonderful',): 1, ('experience',): 1},
 {('worst',): 1, ('purchase',): 1, ('',): 1, ('star',): 1},
 {('pros',): 1, ('cons',): 1, ('',): 1, ('works',): 1, ('perfectly',): 1},
 {('highly',): 1,
  ('recommend',): 1,
  ('',): 1,
  ('amazing',): 1,
  ('features',): 1},
 {('waste',): 1, ('money',): 1, ('',): 1, ('poor',): 1, ('quality',): 1},
 {('love',): 1,
  ('product',): 1,
  ('',): 1,
  ('highly',): 1,
  ('recommend',): 1,
  ('special',): 1},
 {('poor',): 1, ('quality',): 1, ('',): 1, ('worst',): 1, ('purchase',): 1},
 {('awful',): 1,
  ('',): 1,
  ('customer',): 1,
  ('service',): 1,
  ('bad',): 1,
  ('pros',): 1,
  ('cons',): 1},
 {('fantastic',): 1, ('',): 1, ('best',): 1, ('purchase',): 1, ('ever',): 1},
 {('wonderful',): 1,
  ('experience',): 1,
  ('',): 1,
  ('

In [115]:
train

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens,ug
0,REV2000,Product_E,User_114,4,fantastic wonderful experience,2023-04-17,"[fantastic, , wonderful, experience, ]","{('fantastic',): 1, ('',): 2, ('wonderful',): ..."
1,REV2001,Product_C,User_186,2,broke easily awful,2023-11-27,"[broke, easily, , awful, ]","{('broke',): 1, ('easily',): 1, ('',): 2, ('aw..."
2,REV2002,Product_E,User_101,3,met expectations stars,2023-12-10,"[met, expectations, , stars, ]","{('met',): 1, ('expectations',): 1, ('',): 2, ..."
3,REV2003,Product_A,User_175,5,satisfied wonderful experience,2023-11-10,"[satisfied, , wonderful, experience, ]","{('satisfied',): 1, ('',): 2, ('wonderful',): ..."
4,REV2004,Product_C,User_158,1,worst purchase star,2024-05-25,"[worst, purchase, , star, ]","{('worst',): 1, ('purchase',): 1, ('',): 2, ('..."
...,...,...,...,...,...,...,...,...
795,REV2795,Product_E,User_157,4,exceeded expectations best purchase ever,2023-12-30,"[exceeded, expectations, , best, purchase, eve...","{('exceeded',): 1, ('expectations',): 1, ('',)..."
796,REV2796,Product_B,User_167,3,better star,2023-04-08,"[better, , star, ]","{('better',): 1, ('',): 2, ('star',): 1}"
797,REV2797,Product_A,User_156,2,difficult use broke easily,2023-03-26,"[difficult, use, , broke, easily, ]","{('difficult',): 1, ('use',): 1, ('',): 2, ('b..."
798,REV2798,Product_D,User_150,4,love product amazing features met expectations,2024-09-22,"[love, product, , amazing, features, , met, ex...","{('love',): 1, ('product',): 1, ('',): 3, ('am..."


In [116]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=True)
train_sparse_matrix = dv.fit_transform(feature_dict_list)

In [117]:
train_sparse_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3909 stored elements and shape (800, 54)>

**Train to predict category**

In [118]:
dev['ug'] = (dev['tokens']).apply(unigram)

In [119]:
dev

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens,ug
800,REV2800,Product_E,User_102,2,worst purchase star works expected,2023-12-05,"[worst, purchase, , star, , works, expected, ]","{('worst',): 1, ('purchase',): 1, ('',): 3, ('..."
801,REV2801,Product_E,User_192,4,works perfectly fantastic special,2023-05-27,"[works, perfectly, , fantastic, , special, ]","{('works',): 1, ('perfectly',): 1, ('',): 3, (..."
802,REV2802,Product_C,User_184,4,wonderful experience fantastic,2024-03-26,"[wonderful, experience, , fantastic, ]","{('wonderful',): 1, ('experience',): 1, ('',):..."
803,REV2803,Product_C,User_108,5,wonderful experience great value good bad,2023-11-30,"[wonderful, experience, , great, value, , good...","{('wonderful',): 1, ('experience',): 1, ('',):..."
804,REV2804,Product_A,User_121,2,terrible product waste money,2024-02-12,"[terrible, product, , waste, money, ]","{('terrible',): 1, ('product',): 1, ('',): 2, ..."
...,...,...,...,...,...,...,...,...
995,REV2995,Product_D,User_133,3,average product worst purchase,2024-12-10,"[average, product, , worst, purchase, ]","{('average',): 1, ('product',): 1, ('',): 2, (..."
996,REV2996,Product_B,User_143,3,met expectations poor quality,2023-03-19,"[met, expectations, , poor, quality, ]","{('met',): 1, ('expectations',): 1, ('',): 2, ..."
997,REV2997,Product_C,User_185,3,works expected stars,2023-03-15,"[works, expected, , stars, ]","{('works',): 1, ('expected',): 1, ('',): 2, ('..."
998,REV2998,Product_A,User_106,5,fantastic excellent quality good bad,2023-09-26,"[fantastic, , excellent, quality, , good, bad, ]","{('fantastic',): 1, ('',): 3, ('excellent',): ..."


In [123]:
# instance of vectorizer
vectorizer1 = DictVectorizer(sparse=True)

# make binary feature dictionary of train/dev data to make it more suitable
feature_dic_train1 = train['ug'].apply(featurize)
feature_dic_dev1 = dev['ug'].apply(featurize)

# convert to feature vectors
feature_vec_train1 = vectorizer1.fit_transform(feature_dic_train1)
feature_vec_dev1 = vectorizer1.transform(feature_dic_dev1)

# instance of Logistic Regression Model
classifier1 = LogisticRegression()

# Train the model
classifier1.fit(feature_vec_train1, train['ProductID'])

# Predict categories
train['predicted_category'] = classifier1.predict(feature_vec_train1)
dev['predicted_category'] = classifier1.predict(feature_vec_dev1)


In [124]:
train['predicted_category'] = classifier1.predict(feature_vec_train1)

In [125]:
train['predicted_category']

,predicted_category
0,Product_B
1,Product_D
2,Product_D
3,Product_A
4,Product_B
...,...
795,Product_C
796,Product_C
797,Product_D
798,Product_C


In [126]:
train.head()

,ReviewID,ProductID,UserID,Rating,ReviewText,ReviewDate,tokens,ug,predicted_category
0,REV2000,Product_E,User_114,4,fantastic wonderful experience,2023-04-17,"[fantastic, , wonderful, experience, ]","{('fantastic',): 1, ('',): 2, ('wonderful',): ...",Product_B
1,REV2001,Product_C,User_186,2,broke easily awful,2023-11-27,"[broke, easily, , awful, ]","{('broke',): 1, ('easily',): 1, ('',): 2, ('aw...",Product_D
2,REV2002,Product_E,User_101,3,met expectations stars,2023-12-10,"[met, expectations, , stars, ]","{('met',): 1, ('expectations',): 1, ('',): 2, ...",Product_D
3,REV2003,Product_A,User_175,5,satisfied wonderful experience,2023-11-10,"[satisfied, , wonderful, experience, ]","{('satisfied',): 1, ('',): 2, ('wonderful',): ...",Product_A
4,REV2004,Product_C,User_158,1,worst purchase star,2024-05-25,"[worst, purchase, , star, ]","{('worst',): 1, ('purchase',): 1, ('',): 2, ('...",Product_B


In [128]:
lm = LogisticRegression()
lm.fit(train_sparse_matrix, train['Rating'])

LogisticRegression()

In [ ]:
predictions = lm.predict(dev_sparse_matrix)

**Train to predict polarity**

In [130]:
vectorizer1 = DictVectorizer(sparse=True)

# make binary feature dictionary of train/dev data
feature_dic_train1 = train['ug'].apply(featurize)
feature_dic_dev1 = dev['ug'].apply(featurize)

# convert to feature vectors (use .fit_transform/.transform)
feature_vec_train1 = vectorizer1.fit_transform(feature_dic_train1)
feature_vec_dev1 = vectorizer1.transform(feature_dic_dev1)

# instance of Logistic Regression Model
classifier1 = LogisticRegression()

# train model with train data
classifier1.fit(feature_vec_train1, train['Rating'])

LogisticRegression()

In [131]:
predict_result_Rating = classifier1.predict(feature_vec_dev1)

In [132]:
!python3 evaluate.py train.csv pred.csv

python3: can't open file '/content/evaluate.py': [Errno 2] No such file or directory
